# Stage 5 — Master House PTR Workflow: Explain and Run All Four Stages From One Colab

This notebook is the **control panel** for the four-stage House PTR pipeline.

It does not replace the four stage notebooks. Instead, it reads their `.ipynb` files and executes their code cells **in order inside this same Colab runtime**.

## The complete workflow

```text
┌──────────────────────────────────────────────────────────────┐
│ STAGE 1                                                      │
│ Download official House PTR PDFs from the annual XML index   │
└──────────────────────────────┬───────────────────────────────┘
                               ↓
            Congressional Trading Data/House_PTRs/<YEAR>/<DocID>.pdf
                               ↓
┌──────────────────────────────────────────────────────────────┐
│ STAGE 2                                                      │
│ Archive official House XML + verify PDF archive completeness │
└──────────────────────────────┬───────────────────────────────┘
                               ↓
        verified source archive + XML/Excel audit
                               ↓
┌──────────────────────────────────────────────────────────────┐
│ STAGE 3                                                      │
│ Extract transaction data from born-digital PTR PDFs (V8.1)   │
└──────────────────────────────┬───────────────────────────────┘
                               ↓
       V8.1 transactions + scan/image fallback list
                               ↓
┌──────────────────────────────────────────────────────────────┐
│ STAGE 4                                                      │
│ Resolve/clean ticker edge cases from the V8.1 CSV (V8.2)     │
└──────────────────────────────┬───────────────────────────────┘
                               ↓
          final born-digital V8.2 transaction CSV
```


## Project root and filesystem

All stages now use one top-level Google Drive project folder:

```text
MyDrive/
└── Congressional Trading Data/House_PTRs/
    ├── 01 Official House PTR PDFs/
    │   ├── 2021/
    │   ├── 2022/
    │   ├── 2023/
    │   ├── 2024/
    │   ├── 2025/
    │   └── 2026/
    ├── 02 Official House XML Indexes/
    ├── 03 PDF Archive Verification Reports/
    ├── 04 Parsed PTR Transaction Data/
    ├── 05 Parser Checkpoints and Status/
    └── 06 Workflow Notebooks/
```

The master creates these directories automatically if they do not already exist.

This structure keeps every House-data artifact under one recognizable project root while preserving the working internal paths used by the four-stage pipeline.

## Why four stages instead of one giant notebook?

Each stage has one job and one type of evidence:

| Stage | Question it answers | Slow? | Opens PDFs? |
|---|---|---:|---:|
| 1 | What PTR PDFs should I download? | network-dependent | No |
| 2 | Is my source archive complete vs. official House XML? | quick/moderate | No |
| 3 | What transactions are actually inside the born-digital PDFs? | **Yes** | **Yes** |
| 4 | Did ticker interpretation create cleanup edge cases? | fast | No |

This separation makes failures much easier to diagnose and lets Stage 4 improve without rerunning Stage 3.

## Important scope

The four-stage system produces the **born-digital** House transaction dataset.

Stage 3 separately lists scan/image-only PDFs in:

`PTR_GEOMETRY_V8_1_needs_fallback_2021_2026.csv`

Those files still require a future OCR/vision fallback pipeline and are intentionally not hidden or guessed at here.

## One-time setup

Put the **four numbered stage notebooks** in this Drive folder:

```text
MyDrive/
└── Congressional Trading Data/House_PTRs/
    └── 06 Workflow Notebooks/
        ├── 1_Download_Official_House_PTR_PDFs_From_XML_Index.ipynb
        ├── 2_Archive_House_XML_Indexes_And_Verify_PDF_Completeness.ipynb
        ├── 3_Extract_PTR_Transaction_Data_From_Born_Digital_PDFs_V8_1.ipynb
        └── 4_Clean_And_Resolve_PTR_Tickers_From_V8_1_CSV_V8_2.ipynb
```

You can change `WORKFLOW_NOTEBOOK_FOLDER` below if you prefer another folder.

### Is a “notebook that runs other notebooks” actually possible?

Yes.

Colab does not provide a special graphical pipeline button for this, so this master notebook implements one directly:

1. read each `.ipynb` file as JSON
2. take its code cells in order
3. execute each code cell through Colab/IPython's current runtime
4. stop if a child cell errors
5. move to the next stage only after the current stage finishes

This means `%pip` cells, ordinary Python cells, `display()`, and the existing Drive-based checkpoint logic continue to work.

In [ ]:
# =============================================================================
# MASTER SETUP — MOUNT GOOGLE DRIVE
# =============================================================================
# The master mounts Drive before executing any child notebook.
#
# The child notebooks also contain their own drive.mount() cells because each
# remains independently runnable. Calling drive.mount() again after the master
# has mounted it is harmless; Colab normally reports that Drive is already
# mounted.

from google.colab import drive
drive.mount("/content/drive")

## Initialize the Congressional Trading Data/House_PTRs filesystem

This cell is safe to rerun.

`mkdir(..., exist_ok=True)` creates missing folders but does not erase, replace, or modify files already inside them.

In [ ]:
# =============================================================================
# MASTER FILESYSTEM SETUP — CREATE THE DESCRIPTIVE HOUSE DATA FOLDERS
# =============================================================================
#
# Everything is grouped under:
#
#   MyDrive/Congressional Trading Data/House_PTRs/
#
# The numbered folders describe WHAT THEY CONTAIN rather than parser internals.
# mkdir(..., exist_ok=True) is safe to rerun and does not erase existing files.

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Congressional Trading Data/House_PTRs"
)

PDF_ROOT = PROJECT_ROOT / "01 Official House PTR PDFs"
XML_ROOT = PROJECT_ROOT / "02 Official House XML Indexes"
VERIFICATION_ROOT = PROJECT_ROOT / "03 PDF Archive Verification Reports"
DATA_ROOT = PROJECT_ROOT / "04 Parsed PTR Transaction Data"
STATUS_ROOT = PROJECT_ROOT / "05 Parser Checkpoints and Status"
WORKFLOW_NOTEBOOK_FOLDER = PROJECT_ROOT / "06 Workflow Notebooks"

YEARS = list(range(2021, 2027))

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

for folder in [
    PDF_ROOT,
    XML_ROOT,
    VERIFICATION_ROOT,
    DATA_ROOT,
    STATUS_ROOT,
    WORKFLOW_NOTEBOOK_FOLDER,
]:
    folder.mkdir(parents=True, exist_ok=True)

for year in YEARS:
    (PDF_ROOT / str(year)).mkdir(
        parents=True,
        exist_ok=True
    )

print("Congressional Trading Data/House_PTRs filesystem is ready:")
print(PROJECT_ROOT)
print()
print("  01 Official House PTR PDFs/")
for year in YEARS:
    print(f"    {year}/")
print("  02 Official House XML Indexes/")
print("  03 PDF Archive Verification Reports/")
print("  04 Parsed PTR Transaction Data/")
print("  05 Parser Checkpoints and Status/")
print("  06 Workflow Notebooks/")


## One-time move if you already have the old folder layout

The new notebooks expect these descriptive folders.

If your existing finished data is still organized as:

```text
Congressional Trading Data/House_PTRs/
├── 2021/ ... 2026/
├── Original_XML/
├── Indexes/
├── PTR_CSV/
├── V8_1_Status/
└── Workflow_Notebooks/
```

move the existing contents once into:

```text
Congressional Trading Data/House_PTRs/
├── 01 Official House PTR PDFs/
├── 02 Official House XML Indexes/
├── 03 PDF Archive Verification Reports/
├── 04 Parsed PTR Transaction Data/
├── 05 Parser Checkpoints and Status/
└── 06 Workflow Notebooks/
```

Do not delete the old data until you have confirmed the files are present in the new locations.

In [ ]:
# =============================================================================
# MASTER CONFIGURATION — WHERE THE FOUR STAGE NOTEBOOKS LIVE
# =============================================================================
# Change only WORKFLOW_NOTEBOOK_FOLDER if you store the four stage notebooks
# somewhere else.
#
# The booleans let you selectively rerun stages. For a normal full pipeline
# run, leave all four True.

from pathlib import Path
import json
import time
import traceback

WORKFLOW_NOTEBOOK_FOLDER = Path(
    "/content/drive/MyDrive/Congressional Trading Data/House_PTRs/06 Workflow Notebooks"
)

STAGES = [
    {
        "number": 1,
        "title": "Download official House PTR PDFs from XML index",
        "filename": "1_Download_Official_House_PTR_PDFs_From_XML_Index.ipynb",
        "run": True,
    },
    {
        "number": 2,
        "title": "Archive official House XML and verify PDF completeness",
        "filename": "2_Archive_House_XML_Indexes_And_Verify_PDF_Completeness.ipynb",
        "run": True,
    },
    {
        "number": 3,
        "title": "Extract born-digital PTR transaction data with V8.1",
        "filename": "3_Extract_PTR_Transaction_Data_From_Born_Digital_PDFs_V8_1.ipynb",
        "run": True,
    },
    {
        "number": 4,
        "title": "Resolve ticker edge cases and produce V8.2",
        "filename": "4_Clean_And_Resolve_PTR_Tickers_From_V8_1_CSV_V8_2.ipynb",
        "run": True,
    },
]

# True is safest: if one stage fails, do not blindly continue into stages that
# depend on its output.
STOP_ON_ERROR = True

print("Workflow notebook folder:")
print(WORKFLOW_NOTEBOOK_FOLDER)
print()

for stage in STAGES:
    path = WORKFLOW_NOTEBOOK_FOLDER / stage["filename"]
    print(
        f"Stage {stage['number']}:",
        "FOUND" if path.exists() else "MISSING",
        "-",
        stage["filename"]
    )

## Preflight check

Run this before the pipeline.

It deliberately refuses to start when a selected child notebook is missing. That is better than discovering halfway through the workflow that Stage 3 or Stage 4 was not available.

In [ ]:
# =============================================================================
# MASTER PREFLIGHT — REQUIRE EVERY SELECTED NOTEBOOK TO EXIST
# =============================================================================

missing = []

for stage in STAGES:
    if not stage["run"]:
        continue

    path = (
        WORKFLOW_NOTEBOOK_FOLDER
        / stage["filename"]
    )

    if not path.exists():
        missing.append(path)

if missing:
    print("Missing required workflow notebook(s):")
    for path in missing:
        print(" -", path)

    raise FileNotFoundError(
        "\nPut the four numbered stage notebooks in "
        f"{WORKFLOW_NOTEBOOK_FOLDER} before running the master."
    )

print("Preflight passed.")
print("Every selected stage notebook exists.")

## How the master executes a child notebook

The function below intentionally executes **code cells only**.

Markdown cells are documentation and therefore do not need to run.

Each child code cell runs through `get_ipython().run_cell(...)`, which is important because Stage 3 contains a `%pip` magic command that ordinary Python `exec()` would not understand.

The function checks IPython's execution result after every cell. If a cell fails, the master raises an error with the stage/cell number instead of continuing with incomplete data.

In [ ]:
# =============================================================================
# MASTER EXECUTOR — RUN ONE .IPYNB INSIDE THE CURRENT COLAB RUNTIME
# =============================================================================

def run_notebook_in_current_runtime(notebook_path, stage_number, stage_title):
    """
    Execute the code cells from one notebook in this current Colab runtime.

    Why current-runtime execution is useful here:
    - the Drive mount is shared
    - variables/packages installed by earlier cells are immediately available
    - Stage 3's resumable checkpoint logic behaves exactly as when the notebook
      is run directly
    - IPython/Colab magics such as %pip still work
    """

    notebook_path = Path(notebook_path)

    print()
    print("=" * 90)
    print(f"STARTING STAGE {stage_number}: {stage_title}")
    print(notebook_path)
    print("=" * 90)

    # Read the .ipynb file. A notebook is JSON containing ordered cell objects.
    with notebook_path.open(
        "r",
        encoding="utf-8"
    ) as handle:
        notebook = json.load(handle)

    # Keep only executable code cells. Markdown is explanatory documentation.
    code_cells = [
        cell
        for cell in notebook.get("cells", [])
        if cell.get("cell_type") == "code"
    ]

    print(
        f"Executable code cells in Stage {stage_number}:",
        len(code_cells)
    )

    stage_started = time.time()

    for code_index, cell in enumerate(
        code_cells,
        start=1
    ):
        source = "".join(
            cell.get("source", [])
        )

        # Empty code cells do not need to be sent to IPython.
        if not source.strip():
            continue

        print()
        print(
            f"[Stage {stage_number}] "
            f"Running code cell {code_index}/{len(code_cells)}"
        )

        # run_cell() understands ordinary Python plus Colab/IPython magics.
        result = get_ipython().run_cell(
            source,
            store_history=False
        )

        # IPython normally captures exceptions in the ExecutionResult rather
        # than automatically raising them back to this outer loop.
        error = (
            getattr(result, "error_before_exec", None)
            or getattr(result, "error_in_exec", None)
        )

        if error is not None:
            raise RuntimeError(
                f"Stage {stage_number} failed in child code cell "
                f"{code_index}: {error}"
            ) from error

    elapsed_minutes = (
        time.time() - stage_started
    ) / 60.0

    print()
    print("=" * 90)
    print(
        f"FINISHED STAGE {stage_number}: {stage_title}"
    )
    print(
        f"Elapsed: {elapsed_minutes:.1f} minutes"
    )
    print("=" * 90)

# Run the full pipeline

This is the one cell that actually starts the selected stages.

### Normal behavior on a rerun

- **Stage 1:** existing PDFs are skipped.
- **Stage 2:** official XML/audit files are refreshed.
- **Stage 3:** V8.1 reads its checkpoint and skips completed PDFs.
- **Stage 4:** V8.2 is regenerated from the current V8.1 CSV.

That makes rerunning the master much safer than a typical one-shot scraper.

### Runtime interruption

Stage 3 is the long stage. If Colab disconnects during Stage 3, rerun the master later. Stage 3's checkpoint is specifically designed to resume instead of starting the entire archive over.

In [ ]:
# =============================================================================
# MASTER RUNNER — EXECUTE THE SELECTED STAGES IN ORDER
# =============================================================================

workflow_started = time.time()
completed_stages = []

for stage in STAGES:
    if not stage["run"]:
        print(
            f"Skipping Stage {stage['number']}: "
            f"{stage['title']}"
        )
        continue

    notebook_path = (
        WORKFLOW_NOTEBOOK_FOLDER
        / stage["filename"]
    )

    try:
        run_notebook_in_current_runtime(
            notebook_path=notebook_path,
            stage_number=stage["number"],
            stage_title=stage["title"],
        )

        completed_stages.append(
            stage["number"]
        )

    except Exception as error:
        print()
        print("!" * 90)
        print(
            f"STAGE {stage['number']} FAILED"
        )
        print(stage["title"])
        print(repr(error))
        print("!" * 90)

        if STOP_ON_ERROR:
            raise

total_minutes = (
    time.time() - workflow_started
) / 60.0

print()
print("#" * 90)
print("HOUSE PTR WORKFLOW FINISHED")
print("Completed stages:", completed_stages)
print(f"Total elapsed: {total_minutes:.1f} minutes")
print("#" * 90)

## Final output locations

After a complete four-stage run, the important files are:

### Source archive

```text
MyDrive/Congressional Trading Data/House_PTRs/01 Official House PTR PDFs/<YEAR>/<DocID>.pdf
```

### Official XML archive + verification

```text
MyDrive/Congressional Trading Data/House_PTRs/02 Official House XML 03 PDF Archive Verification Reports/<YEAR>FD.xml
MyDrive/Congressional Trading Data/House_PTRs/03 PDF Archive Verification Reports/House_PTR_Verification_<YEAR>.xlsx
```

### Stage 3 born-digital extraction

```text
MyDrive/Congressional Trading Data/House_PTRs/04 Parsed PTR Transaction Data/PTR_transactions_GEOMETRY_V8_1_2021_2026.csv
MyDrive/Congressional Trading Data/House_PTRs/04 Parsed PTR Transaction Data/PTR_GEOMETRY_V8_1_needs_fallback_2021_2026.csv
MyDrive/Congressional Trading Data/House_PTRs/04 Parsed PTR Transaction Data/PTR_GEOMETRY_V8_1_2021_2026.xlsx
```

### Stage 4 cleaned born-digital dataset

```text
MyDrive/Congressional Trading Data/House_PTRs/04 Parsed PTR Transaction Data/PTR_transactions_GEOMETRY_V8_2_2021_2026.csv
MyDrive/Congressional Trading Data/House_PTRs/04 Parsed PTR Transaction Data/PTR_GEOMETRY_V8_2_ticker_QA_2021_2026.csv
```

For analysis, **`PTR_transactions_GEOMETRY_V8_2_2021_2026.csv` is the final born-digital transaction table.**

The `needs_fallback` CSV is not an error to hide. It is the explicit queue of image-only/scanned PDFs that still need OCR/vision extraction.